# Telecom Customer Churn Analysis

**Name:** Sparsh Sharma  
**Admission No.:** 2026B02310041  
**Project No.:** 6  
**Dataset:** `06_customer_churn_expanded_1000.csv`  
**GitHub Link:** `PASTE_YOUR_GITHUB_REPOSITORY_LINK_HERE`

This project analyses telecom customer data to understand churn patterns. The original teaching dataset contained 500 records, and 500 additional synthetic records were generated for learning, giving 1,000 records in total.

In [1]:
# Name: Sparsh Sharma
# Admission No.: 2026B02310041
# Project No.: 6
# Project: Telecom Customer Churn Analysis
# GitHub Link: PASTE_YOUR_GITHUB_REPOSITORY_LINK_HERE

## 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the Unclean Dataset

`keep_default_na=False` preserves the text value `None` in `Internet_Service`. Blank monthly charges are then converted to `NaN` so they can be detected and cleaned.

In [3]:
file_name = "06_customer_churn_expanded_1000.csv"

df = pd.read_csv(file_name, keep_default_na=False)
df["Monthly_Charges"] = pd.to_numeric(df["Monthly_Charges"], errors="coerce")

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
df.head()

Dataset loaded successfully.
Dataset shape: (1000, 8)


,Customer_ID,Tenure_Months,Contract,Monthly_Charges,Internet_Service,Payment_Method,Support_Tickets,Churn
0,CU0001,66,Monthly,896.26,Fiber,Bank Transfer,3,No
1,CU0002,4,Two Year,446.95,DSL,Bank Transfer,2,No
2,CU0003,45,Monthly,492.76,DSL,Card,2,No
3,CU0004,32,One Year,205.23,Fiber,Cash,0,Yes
4,CU0005,67,Monthly,720.44,DSL,Bank Transfer,2,No


## 3. Dataset Inspection

In [4]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print()
print("Column names:")
for column in df.columns:
    print(column)

Number of rows: 1000
Number of columns: 8

Column names:
Customer_ID
Tenure_Months
Contract
Monthly_Charges
Internet_Service
Payment_Method
Support_Tickets
Churn


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Customer_ID       1000 non-null   object 
 1   Tenure_Months     1000 non-null   int64  
 2   Contract          1000 non-null   object 
 3   Monthly_Charges   984 non-null    float64
 4   Internet_Service  1000 non-null   object 
 5   Payment_Method    1000 non-null   object 
 6   Support_Tickets   1000 non-null   int64  
 7   Churn             1000 non-null   object 
dtypes: float64(1), int64(2), object(5)
memory usage: 62.6+ KB


In [6]:
missing_values = df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

Missing values in each column:
Customer_ID          0
Tenure_Months        0
Contract             0
Monthly_Charges     16
Internet_Service     0
Payment_Method       0
Support_Tickets      0
Churn                0
dtype: int64


In [7]:
missing_charge_rows = df[df["Monthly_Charges"].isnull()]
print("Customers with missing Monthly_Charges:", len(missing_charge_rows))
missing_charge_rows

Customers with missing Monthly_Charges: 16


,Customer_ID,Tenure_Months,Contract,Monthly_Charges,Internet_Service,Payment_Method,Support_Tickets,Churn
26,CU0027,37,Two Year,NaN,DSL,Cash,6,No
92,CU0093,38,Two Year,NaN,Fiber,Card,1,Yes
233,CU0234,31,One Year,NaN,None,Cash,0,No
251,CU0252,60,Monthly,NaN,DSL,Card,1,No
366,CU0367,6,Monthly,NaN,DSL,Bank Transfer,2,No
388,CU0389,20,Monthly,NaN,DSL,UPI,2,No
392,CU0393,5,One Year,NaN,Fiber,Cash,2,No
405,CU0406,40,Monthly,NaN,DSL,Cash,1,Yes
517,CU0518,39,Two Year,NaN,DSL,Cash,7,No
572,CU0573,32,One Year,NaN,None,Cash,0,No


In [8]:
print("Duplicate complete rows:", df.duplicated().sum())
print("Duplicate Customer_ID values:", df["Customer_ID"].duplicated().sum())

Duplicate complete rows: 0
Duplicate Customer_ID values: 0


In [9]:
category_columns = ["Contract", "Internet_Service", "Payment_Method", "Churn"]
for column in category_columns:
    print()
    print(f"Values in {column}:")
    print(df[column].value_counts(dropna=False))


Values in Contract:
Contract
Monthly     550
One Year    245
Two Year    205
Name: count, dtype: int64

Values in Internet_Service:
Internet_Service
Fiber    482
DSL      357
None     161
Name: count, dtype: int64

Values in Payment_Method:
Payment_Method
Bank Transfer    296
Card             247
UPI              241
Cash             216
Name: count, dtype: int64

Values in Churn:
Churn
No     721
Yes    279
Name: count, dtype: int64


In [10]:
df.describe().round(2)

,Tenure_Months,Monthly_Charges,Support_Tickets
count,1000.00,984.00,1000.00
mean,35.60,746.78,1.52
std,20.86,222.48,1.34
min,1.00,199.00,0.00
25%,18.00,614.47,1.00
50%,36.00,734.62,1.00
75%,53.00,890.40,2.00
max,72.00,1454.05,7.00


## 4. Data Cleaning

The 16 missing values in `Monthly_Charges` are filled using the median because the median is less affected by extreme values than the mean. The `None` internet-service label is renamed to `No Internet` for clarity.

In [11]:
clean_df = df.copy()
median_monthly_charge = clean_df["Monthly_Charges"].median()
print("Median Monthly Charge before rounding:", median_monthly_charge)

clean_df["Monthly_Charges"] = (
    clean_df["Monthly_Charges"]
    .fillna(median_monthly_charge)
    .round(2)
)
clean_df["Internet_Service"] = clean_df["Internet_Service"].replace("None", "No Internet")
print("Cleaning completed.")

Median Monthly Charge before rounding: 734.615
Cleaning completed.


In [12]:
print("FINAL DATA CLEANING VALIDATION")
print("-" * 40)
print("Dataset shape:", clean_df.shape)
print("Total missing values:", clean_df.isnull().sum().sum())
print("Duplicate complete rows:", clean_df.duplicated().sum())
print("Duplicate Customer IDs:", clean_df["Customer_ID"].duplicated().sum())
print("Unique Customer IDs:", clean_df["Customer_ID"].nunique())
print()
print("Internet Service categories:")
print(clean_df["Internet_Service"].value_counts())

FINAL DATA CLEANING VALIDATION
----------------------------------------
Dataset shape: (1000, 8)
Total missing values: 0
Duplicate complete rows: 0
Duplicate Customer IDs: 0


Unique Customer IDs: 1000

Internet Service categories:
Internet_Service
Fiber          482
DSL            357
No Internet    161
Name: count, dtype: int64


In [13]:
changed_charge_rows = pd.DataFrame({
    "Customer_ID": df.loc[df["Monthly_Charges"].isnull(), "Customer_ID"],
    "Before_Cleaning": df.loc[df["Monthly_Charges"].isnull(), "Monthly_Charges"],
    "After_Cleaning": clean_df.loc[df["Monthly_Charges"].isnull(), "Monthly_Charges"]
})
changed_charge_rows

,Customer_ID,Before_Cleaning,After_Cleaning
26,CU0027,NaN,734.62
92,CU0093,NaN,734.62
233,CU0234,NaN,734.62
251,CU0252,NaN,734.62
366,CU0367,NaN,734.62
388,CU0389,NaN,734.62
392,CU0393,NaN,734.62
405,CU0406,NaN,734.62
517,CU0518,NaN,734.62
572,CU0573,NaN,734.62


In [14]:
cleaning_comparison = pd.DataFrame({
    "Check": ["Total Rows", "Total Columns", "Missing Monthly Charges", "Duplicate Rows", "Duplicate Customer IDs"],
    "Before Cleaning": [df.shape[0], df.shape[1], df["Monthly_Charges"].isnull().sum(), df.duplicated().sum(), df["Customer_ID"].duplicated().sum()],
    "After Cleaning": [clean_df.shape[0], clean_df.shape[1], clean_df["Monthly_Charges"].isnull().sum(), clean_df.duplicated().sum(), clean_df["Customer_ID"].duplicated().sum()]
})
cleaning_comparison

,Check,Before Cleaning,After Cleaning
0,Total Rows,1000,1000
1,Total Columns,8,8
2,Missing Monthly Charges,16,0
3,Duplicate Rows,0,0
4,Duplicate Customer IDs,0,0


In [15]:
clean_df.to_csv("06_customer_churn_cleaned_1000.csv", index=False)
print("Cleaned dataset saved as 06_customer_churn_cleaned_1000.csv")

Cleaned dataset saved as 06_customer_churn_cleaned_1000.csv


## 5. Overall Customer Churn Analysis

In [16]:
churn_counts = clean_df["Churn"].value_counts()
total_customers = len(clean_df)
churned_customers = (clean_df["Churn"] == "Yes").sum()
retained_customers = (clean_df["Churn"] == "No").sum()
churn_rate = churned_customers / total_customers * 100
retention_rate = retained_customers / total_customers * 100

churn_summary = pd.DataFrame({
    "Customer_Status": ["Churned", "Retained"],
    "Customer_Count": [churned_customers, retained_customers],
    "Percentage": [round(churn_rate, 2), round(retention_rate, 2)]
})
print(f"Overall churn rate: {churn_rate:.2f}%")
churn_summary

Overall churn rate: 27.90%


,Customer_Status,Customer_Count,Percentage
0,Churned,279,27.9
1,Retained,721,72.1


In [17]:
plt.figure(figsize=(6, 4))
plt.bar(churn_summary["Customer_Status"], churn_summary["Customer_Count"])
plt.title("Customer Churn Status")
plt.xlabel("Status")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

## 6. Churn Analysis by Contract Type

In [18]:
clean_df["Churn_Flag"] = (clean_df["Churn"] == "Yes").astype(int)

contract_analysis = clean_df.groupby("Contract").agg(
    Total_Customers=("Customer_ID", "count"),
    Churned_Customers=("Churn_Flag", "sum")
).reset_index()
contract_analysis["Churn_Percentage"] = (
    contract_analysis["Churned_Customers"] / contract_analysis["Total_Customers"] * 100
).round(2)
contract_analysis = contract_analysis.sort_values("Churn_Percentage", ascending=False)
contract_analysis

,Contract,Total_Customers,Churned_Customers,Churn_Percentage
1,One Year,245,72,29.39
0,Monthly,550,157,28.55
2,Two Year,205,50,24.39


In [19]:
highest_contract = contract_analysis.iloc[0]
print("Contract with highest churn percentage:", highest_contract["Contract"])
print(f"Churn percentage: {highest_contract['Churn_Percentage']:.2f}%")

Contract with highest churn percentage: One Year
Churn percentage: 29.39%


## 7. Customer Tenure Analysis

In [20]:
tenure_by_churn = clean_df.groupby("Churn")["Tenure_Months"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)
tenure_by_churn

,count,mean,median,min,max
Churn,,,,,
No,721,36.11,37.0,1,72
Yes,279,34.29,33.0,1,72


In [21]:
clean_df["Tenure_Group"] = pd.cut(
    clean_df["Tenure_Months"],
    bins=[0, 6, 12, 24, 48, 72],
    labels=["1-6 Months", "7-12 Months", "13-24 Months", "25-48 Months", "49-72 Months"],
    include_lowest=True
)
tenure_analysis = clean_df.groupby("Tenure_Group", observed=False).agg(
    Total_Customers=("Customer_ID", "count"),
    Churned_Customers=("Churn_Flag", "sum"),
    Average_Tenure=("Tenure_Months", "mean")
).reset_index()
tenure_analysis["Churn_Percentage"] = (
    tenure_analysis["Churned_Customers"] / tenure_analysis["Total_Customers"] * 100
).round(2)
tenure_analysis["Average_Tenure"] = tenure_analysis["Average_Tenure"].round(2)
tenure_analysis

,Tenure_Group,Total_Customers,Churned_Customers,Average_Tenure,Churn_Percentage
0,1-6 Months,105,30,3.18,28.57
1,7-12 Months,81,26,9.69,32.10
2,13-24 Months,158,50,18.80,31.65
3,25-48 Months,347,94,37.06,27.09
4,49-72 Months,309,79,60.37,25.57


In [22]:
highest_tenure_churn = tenure_analysis.sort_values("Churn_Percentage", ascending=False).iloc[0]
print("Tenure group with highest churn:", highest_tenure_churn["Tenure_Group"])
print(f"Churn percentage: {highest_tenure_churn['Churn_Percentage']:.2f}%")

Tenure group with highest churn: 7-12 Months
Churn percentage: 32.10%


## 8. Monthly Charges Analysis

In [23]:
charges_by_churn = clean_df.groupby("Churn")["Monthly_Charges"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)
charges_by_churn

,count,mean,median,min,max
Churn,,,,,
No,721,736.57,728.45,199.0,1403.82
Yes,279,772.48,768.87,199.0,1454.05


In [24]:
average_charge_churned = clean_df.loc[clean_df["Churn"] == "Yes", "Monthly_Charges"].mean()
average_charge_retained = clean_df.loc[clean_df["Churn"] == "No", "Monthly_Charges"].mean()
charge_difference = average_charge_churned - average_charge_retained
print("Average charge of churned customers:", round(average_charge_churned, 2))
print("Average charge of retained customers:", round(average_charge_retained, 2))
print("Difference in average charges:", round(charge_difference, 2))

Average charge of churned customers: 772.48
Average charge of retained customers: 736.57
Difference in average charges: 35.92


## 9. Churn Analysis by Internet Service

In [25]:
internet_analysis = clean_df.groupby("Internet_Service").agg(
    Total_Customers=("Customer_ID", "count"),
    Churned_Customers=("Churn_Flag", "sum")
).reset_index()
internet_analysis["Churn_Percentage"] = (
    internet_analysis["Churned_Customers"] / internet_analysis["Total_Customers"] * 100
).round(2)
internet_analysis = internet_analysis.sort_values("Churn_Percentage", ascending=False)
internet_analysis

,Internet_Service,Total_Customers,Churned_Customers,Churn_Percentage
0,DSL,357,108,30.25
2,No Internet,161,43,26.71
1,Fiber,482,128,26.56


## 10. Churn Analysis by Payment Method

In [26]:
payment_analysis = clean_df.groupby("Payment_Method").agg(
    Total_Customers=("Customer_ID", "count"),
    Churned_Customers=("Churn_Flag", "sum")
).reset_index()
payment_analysis["Churn_Percentage"] = (
    payment_analysis["Churned_Customers"] / payment_analysis["Total_Customers"] * 100
).round(2)
payment_analysis = payment_analysis.sort_values("Churn_Percentage", ascending=False)
payment_analysis

,Payment_Method,Total_Customers,Churned_Customers,Churn_Percentage
1,Card,247,73,29.55
3,UPI,241,69,28.63
0,Bank Transfer,296,82,27.70
2,Cash,216,55,25.46


## 11. Support Ticket Analysis

In [27]:
support_analysis = clean_df.groupby("Churn")["Support_Tickets"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)
support_analysis

,count,mean,median,min,max
Churn,,,,,
No,721,1.50,1.0,0,7
Yes,279,1.57,1.0,0,5


In [28]:
ticket_churn = clean_df.groupby("Support_Tickets").agg(
    Total_Customers=("Customer_ID", "count"),
    Churned_Customers=("Churn_Flag", "sum")
).reset_index()
ticket_churn["Churn_Percentage"] = (
    ticket_churn["Churned_Customers"] / ticket_churn["Total_Customers"] * 100
).round(2)
ticket_churn

,Support_Tickets,Total_Customers,Churned_Customers,Churn_Percentage
0,0,241,69,28.63
1,1,323,89,27.55
2,2,244,55,22.54
3,3,113,37,32.74
4,4,45,17,37.78
5,5,19,12,63.16
6,6,11,0,0.00
7,7,4,0,0.00


## 12. NumPy Statistics for Monthly Charges

In [29]:
charges_array = clean_df["Monthly_Charges"].to_numpy()

numpy_statistics = pd.DataFrame({
    "Statistic": ["Mean", "Median", "Standard Deviation", "Minimum", "Maximum", "25th Percentile", "75th Percentile"],
    "Value": [
        np.mean(charges_array), np.median(charges_array), np.std(charges_array),
        np.min(charges_array), np.max(charges_array),
        np.percentile(charges_array, 25), np.percentile(charges_array, 75)
    ]
})
numpy_statistics["Value"] = numpy_statistics["Value"].round(2)
numpy_statistics

,Statistic,Value
0,Mean,746.59
1,Median,734.62
2,Standard Deviation,220.59
3,Minimum,199.00
4,Maximum,1454.05
5,25th Percentile,618.24
6,75th Percentile,884.88


## 13. IQR Outlier Detection

In [30]:
q1 = np.percentile(charges_array, 25)
q3 = np.percentile(charges_array, 75)
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

charge_outliers = clean_df[
    (clean_df["Monthly_Charges"] < lower_limit) |
    (clean_df["Monthly_Charges"] > upper_limit)
].copy()

print("Q1:", round(q1, 2))
print("Q3:", round(q3, 2))
print("IQR:", round(iqr, 2))
print("Lower limit:", round(lower_limit, 2))
print("Upper limit:", round(upper_limit, 2))
print("Number of charge outliers:", len(charge_outliers))
charge_outliers[["Customer_ID", "Monthly_Charges", "Contract", "Internet_Service", "Churn"]].head(20)

Q1: 618.24
Q3: 884.88
IQR: 266.64
Lower limit: 218.28
Upper limit: 1284.85
Number of charge outliers: 22


,Customer_ID,Monthly_Charges,Contract,Internet_Service,Churn
3,CU0004,205.23,One Year,Fiber,Yes
10,CU0011,1309.85,Monthly,No Internet,No
269,CU0270,1294.61,One Year,Fiber,Yes
310,CU0311,1354.20,Monthly,Fiber,No
324,CU0325,1299.61,Two Year,Fiber,No
355,CU0356,199.00,Two Year,Fiber,No
374,CU0375,199.00,One Year,Fiber,Yes
413,CU0414,1454.05,Monthly,DSL,Yes
508,CU0509,199.00,One Year,Fiber,Yes
523,CU0524,199.00,Monthly,Fiber,No


## 14. Customer Segmentation and GroupBy Summary

In [31]:
clean_df["Charge_Group"] = np.select(
    [
        clean_df["Monthly_Charges"] < q1,
        clean_df["Monthly_Charges"] <= q3
    ],
    ["Low Charge", "Medium Charge"],
    default="High Charge"
)

segment_summary = clean_df.groupby(
    ["Contract", "Charge_Group"], observed=False
).agg(
    Customer_Count=("Customer_ID", "count"),
    Average_Tenure=("Tenure_Months", "mean"),
    Average_Charge=("Monthly_Charges", "mean"),
    Average_Support_Tickets=("Support_Tickets", "mean"),
    Churn_Percentage=("Churn_Flag", lambda x: x.mean() * 100)
).reset_index()

numeric_columns = ["Average_Tenure", "Average_Charge", "Average_Support_Tickets", "Churn_Percentage"]
segment_summary[numeric_columns] = segment_summary[numeric_columns].round(2)
segment_summary

,Contract,Charge_Group,Customer_Count,Average_Tenure,Average_Charge,Average_Support_Tickets,Churn_Percentage
0,Monthly,High Charge,158,36.49,1023.35,1.66,39.87
1,Monthly,Low Charge,117,31.91,478.96,1.50,25.64
2,Monthly,Medium Charge,275,36.85,738.78,1.76,23.27
3,One Year,High Charge,51,33.94,1075.39,1.69,39.22
4,One Year,Low Charge,78,36.74,471.52,1.24,39.74
5,One Year,Medium Charge,116,33.50,736.03,1.14,18.10
6,Two Year,High Charge,41,35.61,1014.29,1.15,9.76
7,Two Year,Low Charge,55,30.82,466.66,1.22,25.45
8,Two Year,Medium Charge,109,39.73,747.16,1.55,29.36


## 15. Key Findings and Recommendations

In [32]:
# Values used for automatically generated findings
highest_contract = contract_analysis.iloc[0]
highest_tenure = tenure_analysis.sort_values("Churn_Percentage", ascending=False).iloc[0]
highest_internet = internet_analysis.iloc[0]
highest_payment = payment_analysis.iloc[0]

findings = [
    f"The overall churn rate is {churn_rate:.2f}% ({churned_customers} out of {total_customers} customers).",
    f"{highest_contract['Contract']} customers have the highest contract-wise churn rate at {highest_contract['Churn_Percentage']:.2f}%.",
    f"The {highest_tenure['Tenure_Group']} group has the highest tenure-group churn rate at {highest_tenure['Churn_Percentage']:.2f}%.",
    f"{highest_internet['Internet_Service']} has the highest internet-service churn rate at {highest_internet['Churn_Percentage']:.2f}%.",
    f"{highest_payment['Payment_Method']} has the highest payment-method churn rate at {highest_payment['Churn_Percentage']:.2f}%.",
    f"Churned customers pay {charge_difference:.2f} more per month on average than retained customers.",
    f"The IQR method identified {len(charge_outliers)} unusually high or low monthly-charge records."
]

for number, finding in enumerate(findings, start=1):
    print(f"{number}. {finding}")

1. The overall churn rate is 27.90% (279 out of 1000 customers).
2. One Year customers have the highest contract-wise churn rate at 29.39%.
3. The 7-12 Months group has the highest tenure-group churn rate at 32.10%.
4. DSL has the highest internet-service churn rate at 30.25%.
5. Card has the highest payment-method churn rate at 29.55%.
6. Churned customers pay 35.92 more per month on average than retained customers.
7. The IQR method identified 22 unusually high or low monthly-charge records.


### Five Customer-Retention Recommendations

1. **Review the highest-churn contract group:** Provide suitable loyalty benefits, plan comparisons, or renewal support to customers in the contract category with the highest observed churn.
2. **Strengthen early-tenure support:** Give the highest-churn tenure group proactive onboarding, service checks, and support follow-ups.
3. **Investigate the highest-churn internet service:** Review service quality, customer complaints, pricing, and plan suitability for that service group.
4. **Review high monthly charges:** Offer billing reviews or suitable plan alternatives because churned customers have a higher average monthly charge in this dataset.
5. **Use support tickets as an alert:** Customers with repeated support requests should receive priority resolution and follow-up before they decide to leave.

These recommendations are based on patterns in the supplied synthetic dataset. The patterns show association and do not prove the exact cause of churn.

## 16. Conclusion

The project cleaned and analysed 1,000 synthetic telecom customer records using Python, Pandas and NumPy. Missing monthly charges were filled with the median, the internet-service categories were clarified, and data quality was validated. The analysis calculated overall churn and compared churn across contracts, tenure groups, internet services, payment methods, charges and support tickets. NumPy was used for numerical statistics and IQR-based outlier detection. The final results can help identify customer groups that may need additional support or suitable retention offers.

In [33]:
# Save the final analysis dataset, including derived columns used in this notebook.
clean_df.to_csv("06_customer_churn_final_analysis_1000.csv", index=False)
print("Final analysis dataset saved successfully.")
print("Notebook project completed successfully.")

Final analysis dataset saved successfully.
Notebook project completed successfully.
